<div class="alert alert-warning">Ensure data already preprocessed using harvest_orig.ipynb</div>

<i>Note: the order of processing kwh and kw does not matter

## **2a.** Processing harvest **kwh** data:
Using meter_data.py (md) functions
* Data interpolation to exact 15 minute intervals

<div style="color:blue">Enter input:

In [14]:
# data input csv
data_path = '../output/harvest_orig_250723-251017.csv' # preprocessed harvest data from harvest_orig.ipynb

##########################################################################

# processed kwh data name
name = 'harvest' # for file naming

Imports:

In [15]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import os,sys

sys.path.append(os.path.abspath('..'))
import modules.harvest_kwh as hv_kwh # import self defined module
import modules.file_naming as fn # import self defined module

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Loading/Processing:

In [16]:
# var for file naming
var = 'kwh'
var2 = 'interval_kwh' # for only kwh data on 15 min intervals

# load kwh data
loaded_kwh = hv_kwh.load_kwh(data_path)

# remove bad spike rows before interpolation
clean_kwh = hv_kwh.clean_kwh_spikes(loaded_kwh)

# processesing kwh (interpolation)
processed_kwh = hv_kwh.process_kwh(clean_kwh)

# create copy of dataframe with only kwh data on 15 min intervals
int_processed_kwh = hv_kwh.interval_kwh(processed_kwh)

# create kwh filenames
kwh_filename = fn.make_filename(processed_kwh, name, var, 'csv')
int_kwh_filename = fn.make_filename(processed_kwh, name, var2, 'csv')

# convert (processed kwh/interpolated meter data) dataframe to csv
processed_kwh.to_csv(kwh_filename, index=False)
int_processed_kwh.to_csv(int_kwh_filename, index=False)

---

<div class="alert alert-warning">Extra code in case for future reference bellow:

### Other:
Extra work done for Eileen for data on meters for testing and small view

In [17]:
# list of meters and first timestamp in dataset
df = pd.read_csv('interpolated_meter_data.csv', encoding="utf-8")
print(df['meter_name'].unique())
print(df.head(1))

FileNotFoundError: [Errno 2] No such file or directory: 'interpolated_meter_data.csv'

In [ ]:
# sample of interpolated data for admin_serv_1 2025-09-10
df = pd.read_csv('interpolated_meter_data.csv', encoding="utf-8")
df['datetime'] = pd.to_datetime(df['datetime'])
date = pd.to_datetime('2025-09-10').date()
sample_data = df[(df['datetime'].dt.date == date) & (df['meter_name'] == 'admin_serv_1_mtr')]
sample_data.to_csv('sample_data.csv', index=False)

# sample of interpolated data for biomedical_science_main_a_mtr 2025-09-10
df = pd.read_csv('interpolated_meter_data.csv', encoding='utf-8')

df['datetime'] = pd.to_datetime(df['datetime'])
date = pd.to_datetime('2025-09-10').date()
sample_data = df[(df['datetime'].dt.date == date) 
    & (df['meter_name'] == 'biomedical_science_main_a_mtr') 
    & ((df['interpolated'] == True) | ((df['datetime'].dt.minute % 15 == 0) & (df['datetime'].dt.second == 0)))
    ].drop_duplicates(subset=['datetime', 'meter_name'])

sample_data.to_csv('sample_data.csv', index=False)

In [ ]:
# get just interpolated meter data from all meters on days sept 07-09 2025
df = pd.read_csv('interpolated_meter_data.csv', encoding='utf-8')
df['datetime'] = pd.to_datetime(df['datetime'])

df = df[df['datetime'].dt.date.isin([
    pd.to_datetime('2025-09-07').date(),
    pd.to_datetime('2025-09-08').date(),
    pd.to_datetime('2025-09-09').date()])]
df = df[df['is_exact'] & True]

df.to_csv('sept07-09_kwh.csv', index=False)

In [ ]:
# find min and max timestamp in kwh data
kwh_file = 'harvest_kwh_250723-251017.csv'
df = pd.read_csv(kwh_file, encoding='utf-8')
print("min:", df['datetime'].min(), "max:", df['datetime'].max())